# 📓 Notebook 3 — Post-hoc Calibration: Platt Scaling & Isotonic Regression
**Uncertainty Quantification in ML Classifiers: A Calibration Benchmarking Study**

---
**Works on:** Google Colab · Jupyter Notebook · JupyterLab · VS Code

**Prerequisite:** Notebooks 1 & 2.  
If their output files are missing, this notebook will **automatically regenerate** everything
it needs — no manual intervention required.

**Outputs produced:**
- `data/calib_results.pkl` — calibrated metrics DataFrame (read by Notebook 4)
- `fig_ece_calibration_comparison.png`
- `fig_before_after_reliability.png`

## 0 · Imports & Shared Utilities

All utility functions are defined here so this notebook is fully self-contained.

In [ ]:
import os, sys, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn import datasets as sk_datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})
PALETTE = {
    'Logistic Regression': '#1d9e75',
    'Random Forest':       '#a32d2d',
    'Gradient Boosting':   '#ba7517',
    'SVM (RBF)':           '#185fa5',
    'MLP':                 '#533ab7',
}

# ── Path helpers ──────────────────────────────────────────────────────────────
DATA_DIR = os.path.join(os.getcwd(), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

def data(filename):
    """Absolute path to data/<filename> relative to the working directory."""
    return os.path.join(DATA_DIR, filename)

print(f"Working directory : {os.getcwd()}")
print(f"Data directory    : {DATA_DIR}")
print(f"Python            : {sys.version.split()[0]}")

In [ ]:
# ── Dataset / split helpers (mirrors Notebook 1) ──────────────────────────────

def load_datasets():
    loaders = {
        'breast_cancer': sk_datasets.load_breast_cancer,
        'wine':          sk_datasets.load_wine,
        'diabetes':      sk_datasets.load_diabetes,
    }
    out = {}
    for name, fn in loaders.items():
        raw = fn()
        X = pd.DataFrame(raw.data, columns=raw.feature_names)
        if name == 'diabetes':
            y      = (raw.target > np.median(raw.target)).astype(int)
            tnames = ['low-progression', 'high-progression']
        else:
            y, tnames = raw.target, list(raw.target_names)
        out[name] = {'X': X, 'y': y, 'target_names': tnames}
    return out


def make_splits(X, y, seed=RANDOM_SEED):
    """60 % train | 20 % calib | 20 % test, stratified."""
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.40, random_state=seed, stratify=y)
    X_cal, X_te, y_cal, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp)
    return X_tr, X_cal, X_te, y_tr, y_cal, y_te


def build_and_save_splits():
    """Build stratified splits from raw data and persist to disk."""
    print("  Building splits from scratch...")
    ds     = load_datasets()
    splits = {}
    for name, d in ds.items():
        X, y = d['X'].values, d['y']
        X_tr, X_cal, X_te, y_tr, y_cal, y_te = make_splits(X, y)
        sc = StandardScaler().fit(X_tr)
        splits[name] = {
            'X_train': sc.transform(X_tr), 'y_train': y_tr,
            'X_calib': sc.transform(X_cal), 'y_calib': y_cal,
            'X_test':  sc.transform(X_te),  'y_test':  y_te,
        }
    with open(data('splits.pkl'), 'wb') as f:
        pickle.dump(splits, f)
    with open(data('meta.pkl'), 'wb') as f:
        pickle.dump({k: {'target_names': v['target_names']} for k, v in ds.items()}, f)
    print(f"  Saved {data('splits.pkl')}")
    return splits


def load_splits():
    """Load splits from disk, auto-building if missing."""
    p = data('splits.pkl')
    if not os.path.exists(p):
        print("  splits.pkl not found — auto-generating...")
        return build_and_save_splits()
    with open(p, 'rb') as f:
        return pickle.load(f)


# ── Classifiers ───────────────────────────────────────────────────────────────

def get_classifiers():
    return {
        'Logistic Regression': LogisticRegression(
            C=1.0, max_iter=1000, random_state=RANDOM_SEED),
        'Random Forest':       RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
        'Gradient Boosting':   GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED),
        'SVM (RBF)':           SVC(
            kernel='rbf', C=1.0, gamma='scale',
            probability=True, random_state=RANDOM_SEED),
        'MLP':                 MLPClassifier(
            hidden_layer_sizes=(100, 50), activation='relu',
            max_iter=500, random_state=RANDOM_SEED),
    }


# ── Calibration metrics ───────────────────────────────────────────────────────

def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any():
            continue
        ece += m.sum() / n * abs(float(y_true[m].mean()) - float(y_prob[m].mean()))
    return ece


def compute_mce(y_true, y_prob, n_bins=10):
    """Maximum Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    mce = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any():
            continue
        mce = max(mce, abs(float(y_true[m].mean()) - float(y_prob[m].mean())))
    return mce


def brier_score(y_true, y_prob):
    return float(np.mean((np.array(y_true) - np.array(y_prob)) ** 2))


def reliability_data(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    centers, confs, freqs, sizes = [], [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if not m.any():
            continue
        centers.append((lo + hi) / 2)
        confs.append(float(y_prob[m].mean()))
        freqs.append(float(y_true[m].mean()))
        sizes.append(int(m.sum()))
    return np.array(centers), np.array(confs), np.array(freqs), np.array(sizes)


def get_probs(clf, X, y):
    """Binary → prob of positive class; multi-class → top-label confidence."""
    proba = clf.predict_proba(X)
    if proba.shape[1] == 2:
        return np.array(y), proba[:, 1]
    return (clf.predict(X) == np.array(y)).astype(int), proba.max(axis=1)


# ── Calibrators ───────────────────────────────────────────────────────────────

class PlattScaler:
    """Platt scaling via a 1-D logistic regression on raw probabilities."""
    def __init__(self):
        self.lr = LogisticRegression(C=1.0, solver='lbfgs')
        self._fallback = False

    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True          # degenerate calib set — pass through
        else:
            self._fallback = False
            self.lr.fit(p.reshape(-1, 1), y)
        return self

    def predict_proba(self, p):
        if self._fallback:
            return p
        return self.lr.predict_proba(p.reshape(-1, 1))[:, 1]


class IsotonicScaler:
    """Isotonic regression calibrator."""
    def __init__(self):
        self.ir = IsotonicRegression(out_of_bounds='clip')
        self._fallback = False

    def fit(self, p, y):
        if len(set(map(int, y))) < 2:
            self._fallback = True
        else:
            self._fallback = False
            self.ir.fit(p, y)
        return self

    def predict_proba(self, p):
        if self._fallback:
            return p
        return self.ir.predict(p)


print("✓ All utilities loaded")

## 1 · Load Artefacts

Splits, trained models, and raw probability outputs from Notebooks 1 & 2.  
Any missing file is **automatically rebuilt** — no need to re-run earlier notebooks manually.

In [ ]:
# ── Load splits (auto-build if missing) ───────────────────────────────────────
splits = load_splits()
DATASET_NAMES = list(splits.keys())
MODEL_NAMES   = list(get_classifiers().keys())


# ── Helper: load a pkl, running a rebuild function if it doesn't exist ─────────
def _require(filename, rebuild_fn):
    """
    Load `filename` from the data directory.
    If the file is absent, call `rebuild_fn()` first, then load.
    """
    p = data(filename)
    if not os.path.exists(p):
        print(f"  {filename} missing — rebuilding (this may take ~30 s)...")
        rebuild_fn()
    with open(p, 'rb') as f:
        return pickle.load(f)


def _rebuild_models():
    """
    Replicate the work of Notebook 2: train all classifiers and
    write trained_models.pkl, raw_probs.pkl, raw_metrics.pkl.
    """
    print("  Re-training models (Notebook 2 output was missing)...")
    tm, rp, metrics = {}, {}, []
    for ds_name in DATASET_NAMES:
        sp = splits[ds_name]
        tm[ds_name], rp[ds_name] = {}, {}
        for clf_name, clf in get_classifiers().items():
            t0 = time.time()
            clf.fit(sp['X_train'], sp['y_train'])
            elapsed = time.time() - t0
            y_eval, prob_pos = get_probs(clf, sp['X_test'], sp['y_test'])
            tm[ds_name][clf_name] = clf
            rp[ds_name][clf_name] = (y_eval, prob_pos)
            metrics.append({
                'Dataset':  ds_name,
                'Model':    clf_name,
                'Accuracy': round(accuracy_score(sp['y_test'], clf.predict(sp['X_test'])), 4),
                'ECE':      round(compute_ece(y_eval, prob_pos), 4),
                'MCE':      round(compute_mce(y_eval, prob_pos), 4),
                'Brier':    round(brier_score(y_eval, prob_pos), 4),
                'Time_s':   round(elapsed, 2),
            })
            print(f"    {ds_name:<18} {clf_name:<22} "
                  f"Acc={metrics[-1]['Accuracy']:.3f}  "
                  f"ECE={metrics[-1]['ECE']:.4f}  ({elapsed:.1f}s)")
    with open(data('trained_models.pkl'), 'wb') as f: pickle.dump(tm, f)
    with open(data('raw_probs.pkl'),      'wb') as f: pickle.dump(rp, f)
    with open(data('raw_metrics.pkl'),    'wb') as f: pickle.dump(pd.DataFrame(metrics), f)
    print("  Rebuild complete.")


# ── Load (or rebuild) the three artefacts from Notebook 2 ─────────────────────
trained_models = _require('trained_models.pkl', _rebuild_models)
raw_probs      = _require('raw_probs.pkl',      _rebuild_models)
raw_metrics_df = _require('raw_metrics.pkl',    _rebuild_models)

print()
print("✓ Artefacts ready")
print(f"  Datasets : {DATASET_NAMES}")
print(f"  Models   : {MODEL_NAMES}")

## 2 · Calibration Loop

For every (dataset, model) pair:
1. Fit **Platt scaling** (logistic regression on the calibration split probabilities)
2. Fit **Isotonic regression** on the same calibration split
3. Evaluate ECE / MCE / accuracy on the held-out test split before and after calibration

In [ ]:
calib_records = []

for ds_name in DATASET_NAMES:
    sp = splits[ds_name]
    for clf_name in MODEL_NAMES:
        clf = trained_models[ds_name][clf_name]

        # Raw test-set probabilities (produced by Notebook 2)
        y_test_eval,  raw_prob_test  = raw_probs[ds_name][clf_name]

        # Calibration-split probabilities (never seen during training)
        y_calib_eval, raw_prob_calib = get_probs(clf, sp['X_calib'], sp['y_calib'])

        # Fit calibrators on the calib split, evaluate on the test split
        platt      = PlattScaler().fit(raw_prob_calib, y_calib_eval)
        iso        = IsotonicScaler().fit(raw_prob_calib, y_calib_eval)
        platt_prob = platt.predict_proba(raw_prob_test)
        iso_prob   = iso.predict_proba(raw_prob_test)

        raw_ece   = compute_ece(y_test_eval, raw_prob_test)
        platt_ece = compute_ece(y_test_eval, platt_prob)
        iso_ece   = compute_ece(y_test_eval, iso_prob)
        denom     = max(raw_ece, 1e-9)          # avoid division by zero

        calib_records.append({
            'Dataset':   ds_name,
            'Model':     clf_name,
            # ── ECE ─────────────────────────────────────────────────────────
            'Raw_ECE':   round(raw_ece,   4),
            'Platt_ECE': round(platt_ece, 4),
            'Iso_ECE':   round(iso_ece,   4),
            # ── MCE ─────────────────────────────────────────────────────────
            'Raw_MCE':   round(compute_mce(y_test_eval, raw_prob_test), 4),
            'Iso_MCE':   round(compute_mce(y_test_eval, iso_prob),      4),
            # ── Accuracy ────────────────────────────────────────────────────
            'Raw_acc':   round(accuracy_score(sp['y_test'], clf.predict(sp['X_test'])), 4),
            'Platt_acc': round(accuracy_score(y_test_eval, (platt_prob >= 0.5).astype(int)), 4),
            'Iso_acc':   round(accuracy_score(y_test_eval, (iso_prob   >= 0.5).astype(int)), 4),
            # ── % change in ECE (negative = improvement) ─────────────────
            'Platt_pct': round((platt_ece - raw_ece) / denom * 100, 1),
            'Iso_pct':   round((iso_ece   - raw_ece) / denom * 100, 1),
            # ── Raw arrays (used by reliability diagram plots) ────────────
            '_y':     y_test_eval,
            '_raw':   raw_prob_test,
            '_platt': platt_prob,
            '_iso':   iso_prob,
        })
        print(f"  {ds_name:<18} {clf_name:<22}  "
              f"ECE {raw_ece:.4f}  →  Platt {platt_ece:.4f}  →  Iso {iso_ece:.4f}")

calib_df = pd.DataFrame(calib_records)
print(f"\n✓ {len(calib_df)} combinations calibrated")

## 3 · ECE Reduction Summary

Mean ECE before and after calibration, averaged across all three datasets.

In [ ]:
reduction = (
    calib_df
    .groupby('Model')[['Raw_ECE', 'Platt_ECE', 'Iso_ECE', 'Platt_pct', 'Iso_pct']]
    .mean()
    .round(4)
    .sort_values('Iso_ECE')
)
reduction.columns = ['Raw ECE', 'Platt ECE', 'Isotonic ECE', 'Platt Δ%', 'Iso Δ%']

print("Mean ECE before / after calibration (averaged across 3 datasets):")
display(reduction)

print(f"\nIsotonic  — average ECE Δ : {calib_df['Iso_pct'].mean():.1f}%")
print(f"Platt     — average ECE Δ : {calib_df['Platt_pct'].mean():.1f}%")
print("(Negative values mean improvement.)")

## 4 · ECE Before vs. After — Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ECE Before vs. After Post-hoc Calibration', fontweight='bold')

for ax, ds_name in zip(axes, DATASET_NAMES):
    sub = (
        calib_df[calib_df['Dataset'] == ds_name]
        .sort_values('Raw_ECE', ascending=False)
    )
    x, w = np.arange(len(sub)), 0.25
    ax.bar(x - w, sub['Raw_ECE'],   width=w, label='Raw',      color='#a32d2d', alpha=0.85)
    ax.bar(x,     sub['Platt_ECE'], width=w, label='Platt',    color='#ba7517', alpha=0.85)
    ax.bar(x + w, sub['Iso_ECE'],   width=w, label='Isotonic', color='#1d9e75', alpha=0.85)
    ax.axhline(0.05, color='#533ab7', ls=':', lw=1.4, label='ECE = 0.05')
    ax.set_xticks(x)
    ax.set_xticklabels(
        [m.replace(' ', '\n') for m in sub['Model']], fontsize=8)
    ax.set_ylabel('ECE')
    ax.set_title(ds_name.replace('_', ' ').title())
    if ax is axes[0]:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_ece_calibration_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("→ Saved fig_ece_calibration_comparison.png")

## 5 · Reliability Diagrams: Before vs. After

Shown for **Random Forest on Breast Cancer** — the model with the largest raw ECE
in most runs, making the calibration effect most visible.

In [ ]:
ds_name, clf_name = 'breast_cancer', 'Random Forest'
row = calib_df[
    (calib_df['Dataset'] == ds_name) & (calib_df['Model'] == clf_name)
].iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
fig.suptitle(
    f'{clf_name} — {ds_name.replace("_"," ").title()}: Before & After Calibration',
    fontweight='bold', y=1.03)

configs = [
    ('Raw Predictions',      '_raw',   'Raw_ECE',   '#a32d2d'),
    ('After Platt Scaling',  '_platt', 'Platt_ECE', '#ba7517'),
    ('After Isotonic Regr.', '_iso',   'Iso_ECE',   '#1d9e75'),
]

for ax, (title, prob_key, ece_key, col) in zip(axes, configs):
    _, confs, freqs, _ = reliability_data(row['_y'], row[prob_key])
    ax.fill_between(confs, confs, freqs, alpha=0.15, color=col)
    ax.plot([0, 1], [0, 1], '--', color='#aaa', lw=1.2, zorder=0)
    ax.plot(confs, freqs, 'o-', color=col, ms=7, lw=2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('Mean Confidence')
    if ax is axes[0]:
        ax.set_ylabel('Observed Frequency')
    ax.set_title(f'{title}\nECE = {row[ece_key]:.4f}', fontsize=11)

plt.tight_layout()
plt.savefig('fig_before_after_reliability.png', bbox_inches='tight', dpi=150)
plt.show()
print("→ Saved fig_before_after_reliability.png")

## 6 · Accuracy Preservation Check

Post-hoc calibration should not meaningfully change decision-boundary accuracy
(since it only rescales probabilities, not the ordering of predictions).

In [ ]:
acc = (
    calib_df
    .groupby('Model')[['Raw_acc', 'Platt_acc', 'Iso_acc']]
    .mean()
    .round(4)
)
acc['Iso Δ (pp)'] = ((acc['Iso_acc'] - acc['Raw_acc']) * 100).round(2)
acc.columns = ['Raw Acc', 'Platt Acc', 'Isotonic Acc', 'Iso Δ (pp)']

display(acc)
print(f"Max accuracy drop from isotonic calibration: {acc['Iso Δ (pp)'].min():.2f} pp")

## 7 · Save Results

In [ ]:
# Drop the internal array columns before pickling to keep the file small
save_df = calib_df.drop(columns=['_y', '_raw', '_platt', '_iso'])

with open(data('calib_results.pkl'), 'wb') as f:
    pickle.dump(save_df, f)

# Also persist the full DataFrame (with arrays) for any notebook that needs
# the raw probability vectors directly.
with open(data('calib_results_full.pkl'), 'wb') as f:
    pickle.dump(calib_df, f)

print(f"✓ Saved {data('calib_results.pkl')}")
print(f"✓ Saved {data('calib_results_full.pkl')} (includes probability arrays)")
print()
print("Files in data/:")
for fname in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(os.path.join(DATA_DIR, fname)) / 1024
    print(f"  {fname:<30} {size_kb:6.1f} KB")
print()
print("✓ Notebook 3 complete — run Notebook 4 next")